#MobileNetV2

In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from torchvision import transforms, models
from tqdm import tqdm

# ============================================================
# CONFIGURATION
# ============================================================

DATASET_PATH = "/content/drive/MyDrive/Thesis Dataset Exp/Hands_Only/DATASET_CROPPED_NEW_ADDED"

NUM_FRAMES = 16
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 20
LR = 3e-5

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ============================================================
# LOAD VIDEO PATHS
# ============================================================

def load_video_paths(dataset_path):

    file_paths = []
    labels = []

    class_names = sorted([
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ])

    for label_idx, cls in enumerate(class_names):

        class_dir = os.path.join(dataset_path, cls)

        for angle in ["Side", "Front"]:

            angle_dir = os.path.join(class_dir, angle)

            if not os.path.isdir(angle_dir):
                continue

            for fname in sorted(os.listdir(angle_dir)):

                if fname.lower().endswith((".mp4", ".avi", ".mov")):

                    file_paths.append(os.path.join(angle_dir, fname))
                    labels.append(label_idx)

    return file_paths, labels, class_names


file_paths, labels, class_names = load_video_paths(DATASET_PATH)

print("Total videos:", len(file_paths))
print("Classes:", class_names)

# ============================================================
# SPLIT DATASET (Train / Val / Test)
# ============================================================

trainval_paths, test_paths, trainval_labels, test_labels = train_test_split(
    file_paths,
    labels,
    test_size=0.20,
    stratify=labels,
    random_state=42
)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    trainval_paths,
    trainval_labels,
    test_size=0.125,  # gives ~10% of total dataset
    stratify=trainval_labels,
    random_state=42
)

print("Train:", len(train_paths))
print("Validation:", len(val_paths))
print("Test:", len(test_paths))

# ============================================================
# DATA AUGMENTATION
# ============================================================

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.35, contrast=0.35),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ============================================================
# FRAME SAMPLING
# ============================================================

def sample_frames(video_path, num_frames=NUM_FRAMES):

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total < num_frames:
        frames_idx = list(range(total)) + [total-1]*(num_frames-total)
    else:
        interval = total // num_frames
        frames_idx = [i*interval for i in range(num_frames)]

    frames = []

    for idx in frames_idx:

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()

    return np.array(frames)

# ============================================================
# DATASET
# ============================================================

class VideoDataset(Dataset):

    def __init__(self, paths, labels, transform):

        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        video = self.paths[idx]
        label = self.labels[idx]

        frames = sample_frames(video, NUM_FRAMES)

        processed = []

        for f in frames:
            processed.append(self.transform(f))

        video_tensor = torch.stack(processed)  # (T,C,H,W)

        return video_tensor, label

# ============================================================
# DATASET OBJECTS
# ============================================================

train_dataset = VideoDataset(train_paths, train_labels, train_transform)
val_dataset = VideoDataset(val_paths, val_labels, test_transform)
test_dataset = VideoDataset(test_paths, test_labels, test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# ============================================================
# IMPROVED MOBILENETV2 VIDEO MODEL
# ============================================================

class MobileNetVideoClassifier(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        backbone = models.mobilenet_v2(pretrained=True)

        self.feature_extractor = backbone.features
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(1280, num_classes)
        )

    def forward(self, x):

        B, T, C, H, W = x.shape

        x = x.view(B*T, C, H, W)

        features = self.feature_extractor(x)

        features = self.pool(features)

        features = features.view(B*T, 1280)

        features = features.view(B, T, 1280)

        features = features.mean(dim=1)

        out = self.classifier(features)

        return out

# ============================================================
# MODEL SETUP
# ============================================================

model = MobileNetVideoClassifier(len(class_names)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ============================================================
# TRAINING LOOP
# ============================================================

for epoch in range(EPOCHS):

    model.train()

    correct = 0
    total = 0

    for videos, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):

        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        preds = outputs.argmax(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # VALIDATION
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for videos, labels in val_loader:

            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)

            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total

    print(f"Epoch {epoch+1} Train Acc {train_acc:.4f} Val Acc {val_acc:.4f}")

# ============================================================ # FINAL TEST EVALUATION # ============================================================
model.eval() all_preds = [] all_true = [] correct = 0 total = 0 with torch.no_grad():
for videos, labels in test_loader:
  videos = videos.to(device) labels = labels.to(device)
  outputs = model(videos) preds = outputs.argmax(1)
  correct += (preds == labels).sum().item()
  total += labels.size(0)
  all_preds.extend(preds.cpu().numpy())
  all_true.extend(labels.cpu().numpy())
  test_accuracy = 100 * correct / total
  print("\n==============================")
  print("FINAL TEST RESULT")
  print("==============================")
  print(f"Final Test Accuracy: {test_accuracy:.2f}%")

  # ============================================================ # CONFUSION MATRIX # ============================================================

  cm = confusion_matrix(all_true, all_preds)
  plt.figure(figsize=(10,8))
  sns.heatmap( cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names )
  plt.title("Confusion Matrix")
  plt.ylabel("True Label")
  plt.xlabel("Predicted Label")
  plt.show()

  # ============================================================ # SAVE MODEL # ============================================================
  torch.save(model.state_dict(), "mobilenetv2_video_model.pth")
  print("Model saved: mobilenetv2_video_model.pth")


#EfficientNet

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from torchvision import transforms
import timm
from tqdm import tqdm

# ============================================================
# CONFIGURATION
# ============================================================
DATASET_PATH = "/content/drive/MyDrive/Thesis Dataset Exp/Hands_Only/DATASET_CROPPED_NEW_ADDED"

NUM_FRAMES = 12
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 20
LR = 1e-4

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ============================================================
# LOAD VIDEO PATHS
# ============================================================
def load_video_paths(dataset_path):
    file_paths, labels = [], []

    class_names = sorted([
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ])

    for label_idx, cls in enumerate(class_names):
        class_dir = os.path.join(dataset_path, cls)

        for angle in ["Side", "Front"]:
            angle_dir = os.path.join(class_dir, angle)

            if not os.path.isdir(angle_dir):
                continue

            for fname in sorted(os.listdir(angle_dir)):
                if fname.lower().endswith((".mp4", ".avi", ".mov")):
                    file_paths.append(os.path.join(angle_dir, fname))
                    labels.append(label_idx)

    return file_paths, labels, class_names


file_paths, labels, class_names = load_video_paths(DATASET_PATH)

print("Total videos:", len(file_paths))
print("Classes:", class_names)

# ============================================================
# SPLIT DATASET (70 / 15 / 15)
# ============================================================
trainval_paths, test_paths, trainval_labels, test_labels = train_test_split(
    file_paths, labels,
    test_size=0.15,
    stratify=labels,
    random_state=42
)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    trainval_paths, trainval_labels,
    test_size=0.1765,
    stratify=trainval_labels,
    random_state=42
)

print("Train:", len(train_paths))
print("Val:", len(val_paths))
print("Test:", len(test_paths))

# ============================================================
# TRANSFORMS
# ============================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ============================================================
# FRAME SAMPLING
# ============================================================
def sample_frames(video_path, num_frames=NUM_FRAMES):
    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total < num_frames:
        frames_idx = list(range(total)) + [total-1]*(num_frames-total)
    else:
        interval = total // num_frames
        frames_idx = [i*interval for i in range(num_frames)]

    frames = []
    for idx in frames_idx:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()
        if not ret:
            continue
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()
    return np.array(frames)

# ============================================================
# DATASET
# ============================================================
class VideoDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        video = self.paths[idx]
        label = self.labels[idx]

        frames = sample_frames(video, NUM_FRAMES)
        processed = [self.transform(f) for f in frames]

        video_tensor = torch.stack(processed)  # (T,C,H,W)
        return video_tensor, label

# ============================================================
# DATALOADER
# ============================================================
train_loader = DataLoader(
    VideoDataset(train_paths, train_labels, train_transform),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2
)

val_loader = DataLoader(
    VideoDataset(val_paths, val_labels, test_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

test_loader = DataLoader(
    VideoDataset(test_paths, test_labels, test_transform),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

# ============================================================
# EFFICIENTNET + LSTM MODEL
# ============================================================
class EfficientNetLSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        # Spatial feature extractor
        self.backbone = timm.create_model(
            "efficientnet_b0",
            pretrained=True,
            num_classes=0,
            global_pool="avg"
        )

        self.feature_dim = self.backbone.num_features

        # Temporal modeling
        self.lstm = nn.LSTM(
            input_size=self.feature_dim,
            hidden_size=256,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.BatchNorm1d(512),
            nn.Dropout(0.5),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        B, T, C, H, W = x.shape

        x = x.view(B*T, C, H, W)
        feats = self.backbone(x)        # (B*T, F)
        feats = feats.view(B, T, -1)    # (B, T, F)

        lstm_out, _ = self.lstm(feats)  # (B, T, 512)
        lstm_out = lstm_out[:, -1, :]   # last timestep

        return self.classifier(lstm_out)

# ============================================================
# MODEL SETUP
# ============================================================
model = EfficientNetLSTM(len(class_names)).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR)

# ============================================================
# TRAINING LOOP
# ============================================================
for epoch in range(EPOCHS):

    # TRAIN
    model.train()
    correct, total = 0, 0

    for videos, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        videos, labels = videos.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(videos)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total

    # VALIDATION
    model.eval()
    correct, total = 0, 0

    with torch.no_grad():
        for videos, labels in val_loader:
            videos, labels = videos.to(device), labels.to(device)

            outputs = model(videos)
            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    val_acc = correct / total

    print(f"Epoch {epoch+1}: Train Acc {train_acc:.4f} | Val Acc {val_acc:.4f}")

# ============================================================
# TEST EVALUATION
# ============================================================
model.eval()
all_preds, all_true = [], []

with torch.no_grad():
    for videos, labels in test_loader:
        videos = videos.to(device)

        outputs = model(videos)
        preds = outputs.argmax(1).cpu().numpy()

        all_preds.extend(preds)
        all_true.extend(labels.numpy())

print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=class_names))

# ============================================================
# CONFUSION MATRIX
# ============================================================
cm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.show()

# ============================================================
# SAVE MODEL
# ============================================================
torch.save(model.state_dict(), "efficientnet_lstm_video.pth")
print("Model saved!")

#X3D-S

In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import cv2
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
import matplotlib.pyplot as plt
from pytorchvideo.models.hub import x3d_s
from torchvision import transforms
from tqdm import tqdm

# ============================================================
# CONFIGURATION
# ============================================================
DATASET_PATH = "/content/drive/MyDrive/Thesis Dataset Exp/Hands_Only/DATASET_CROPPED_NEW_ADDED"
NUM_FRAMES = 16
IMG_SIZE = 224
BATCH_SIZE = 4
EPOCHS = 20
LR = 3e-5

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ============================================================
# LOAD VIDEO PATHS
# ============================================================
def load_video_paths(dataset_path):

    file_paths = []
    labels = []

    class_names = sorted([
        d for d in os.listdir(dataset_path)
        if os.path.isdir(os.path.join(dataset_path, d))
    ])

    for label_idx, cls in enumerate(class_names):

        class_dir = os.path.join(dataset_path, cls)

        for angle in ["Side", "Front"]:

            angle_dir = os.path.join(class_dir, angle)

            if not os.path.isdir(angle_dir):
                continue

            for fname in sorted(os.listdir(angle_dir)):

                if fname.lower().endswith((".mp4", ".avi", ".mov")):

                    file_paths.append(os.path.join(angle_dir, fname))
                    labels.append(label_idx)

    return file_paths, labels, class_names


file_paths, labels, class_names = load_video_paths(DATASET_PATH)

print("Total videos:", len(file_paths))
print("Classes:", class_names)

# ============================================================
# TRAIN / VAL / TEST SPLIT
# ============================================================
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    file_paths,
    labels,
    test_size=0.30,
    stratify=labels,
    random_state=42
)

val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

print("Train videos:", len(train_paths))
print("Validation videos:", len(val_paths))
print("Test videos:", len(test_paths))

# ============================================================
# DATA AUGMENTATION
# ============================================================
train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.35, contrast=0.35),
    transforms.ToTensor(),
])

test_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ============================================================
# FRAME SAMPLING
# ============================================================
def sample_frames(video_path, num_frames=NUM_FRAMES):

    cap = cv2.VideoCapture(video_path)
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total < num_frames:
        frames_idx = list(range(total)) + [total - 1] * (num_frames - total)
    else:
        interval = total // num_frames
        frames_idx = [i * interval for i in range(num_frames)]

    frames = []

    for idx in frames_idx:

        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ret, frame = cap.read()

        if not ret:
            continue

        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frames.append(frame)

    cap.release()

    return np.array(frames)

# ============================================================
# DATASET
# ============================================================
class VideoDataset(Dataset):

    def __init__(self, paths, labels, transform):

        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):

        video = self.paths[idx]
        label = self.labels[idx]

        frames = sample_frames(video, NUM_FRAMES)

        processed = []

        for f in frames:
            processed.append(self.transform(f))

        video_tensor = torch.stack(processed)
        video_tensor = video_tensor.permute(1,0,2,3)

        return video_tensor, label

# ============================================================
# DATALOADERS
# ============================================================
train_dataset = VideoDataset(train_paths, train_labels, train_transform)
val_dataset   = VideoDataset(val_paths, val_labels, test_transform)
test_dataset  = VideoDataset(test_paths, test_labels, test_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# ============================================================
# MODEL
# ============================================================
model = x3d_s(pretrained=True)

# model.blocks[5].proj = nn.Linear(
#     model.blocks[5].proj.in_features,
#     len(class_names)
# )

model.blocks[5].proj = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(model.blocks[5].proj.in_features, len(class_names))
)

model = model.to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

# ============================================================
# TRAINING
# ============================================================
history = {
    "train_loss":[],
    "train_acc":[],
    "val_loss":[],
    "val_acc":[]
}

for epoch in range(EPOCHS):

    # ---------------- TRAIN ----------------
    model.train()

    train_loss = 0
    correct = 0
    total = 0

    for videos, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Train"):

        videos = videos.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(videos)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * videos.size(0)

        preds = outputs.argmax(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    epoch_train_loss = train_loss / total
    epoch_train_acc = correct / total

    history["train_loss"].append(epoch_train_loss)
    history["train_acc"].append(epoch_train_acc)

    # ---------------- VALIDATION ----------------
    model.eval()

    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():

        for videos, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} Val"):

            videos = videos.to(device)
            labels = labels.to(device)

            outputs = model(videos)

            loss = criterion(outputs, labels)

            val_loss += loss.item() * videos.size(0)

            preds = outputs.argmax(1)

            correct += (preds == labels).sum().item()
            total += labels.size(0)

    epoch_val_loss = val_loss / total
    epoch_val_acc = correct / total

    history["val_loss"].append(epoch_val_loss)
    history["val_acc"].append(epoch_val_acc)

    print(
        f"Epoch {epoch+1}/{EPOCHS} | "
        f"Train Loss {epoch_train_loss:.4f} Acc {epoch_train_acc:.4f} | "
        f"Val Loss {epoch_val_loss:.4f} Acc {epoch_val_acc:.4f}"
    )

# ============================================================
# TRAINING CURVES
# ============================================================
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history["train_acc"], label="Train")
plt.plot(history["val_acc"], label="Validation")
plt.title("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history["train_loss"], label="Train")
plt.plot(history["val_loss"], label="Validation")
plt.title("Loss")
plt.legend()

plt.show()

# ============================================================
# FINAL TEST EVALUATION
# ============================================================
model.eval()

all_preds = []
all_true = []

correct = 0
total = 0

with torch.no_grad():

    for videos, labels in test_loader:

        videos = videos.to(device)
        labels = labels.to(device)

        outputs = model(videos)

        preds = outputs.argmax(1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.cpu().numpy())

test_accuracy = 100 * correct / total

print("\n==============================")
print("FINAL TEST RESULT")
print("==============================")
print(f"Final Test Accuracy: {test_accuracy:.2f}%")

# ============================================================
# CLASSIFICATION REPORT
# ============================================================
print("\nClassification Report:\n")
print(classification_report(all_true, all_preds, target_names=class_names))

# ============================================================
# CONFUSION MATRIX
# ============================================================
cm = confusion_matrix(all_true, all_preds)

plt.figure(figsize=(10,8))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)

plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")

plt.show()

# ============================================================
# SAVE MODEL
# ============================================================
torch.save(model.state_dict(), "x3d_s_mixed_emotion.pth")

print("Model saved: x3d_s_13_classes_final.pth")